## Reproducing ResNet on CIFAR-10: Experiments on Network Depth, Batch Size, and Pooling

---
- Baseline Code Link: https://github.com/kuangliu/pytorch-cifar

In [1]:
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from torchsummary import summary

import os
import argparse

import matplotlib.pyplot as plt
import numpy as np

# from resnet_20_32_44_56_v1 import *
from resnet_20_32_44_56_v2 import *
from utils import progress_bar

In [2]:
parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
parser.add_argument('--resume', '-r', action='store_true',
                    help='resume from checkpoint')
# args = parser.parse_args()
args, _ = parser.parse_known_args()

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")
best_acc = 0   # best test accuracy
start_epoch = 0   # start from epoch 0 or last checkpoint epoch

device: mps


In [3]:
# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

==> Preparing data..
Files already downloaded and verified
Files already downloaded and verified


In [5]:
# Model
print('==> Building Model..\n')

# net = ResNet20()
net = ResNet32()
# net = ResNet44()
# net = ResNet56()

# Check layer(type), output shape, param #
print('==> Model Summary')
summary(net, (3, 32, 32))

net = net.to(device)

if args.resume:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

criterion = nn.CrossEntropyLoss()
# 'We use a weight decay of 0.0001 and momentum of 0.9' (p.7)
optimizer = optim.SGD(net.parameters(), lr=args.lr,
                      momentum=0.9, weight_decay=0.0001)
# 'We start with a learning rate of 0.1, divide it by 10 at 32k and 48k iterations, and terminate training at 64k iterations' (p.7)
# This code terminates training at the 200 epoch, so the learning rate is divided by 10 at the 100 and 150 epochs to match the same ratio as in the paper
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1)

==> Building Model..

==> Model Summary
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             432
       BatchNorm2d-2           [-1, 16, 32, 32]              32
            Conv2d-3           [-1, 16, 32, 32]           2,304
       BatchNorm2d-4           [-1, 16, 32, 32]              32
            Conv2d-5           [-1, 16, 32, 32]           2,304
       BatchNorm2d-6           [-1, 16, 32, 32]              32
        BasicBlock-7           [-1, 16, 32, 32]               0
            Conv2d-8           [-1, 16, 32, 32]           2,304
       BatchNorm2d-9           [-1, 16, 32, 32]              32
           Conv2d-10           [-1, 16, 32, 32]           2,304
      BatchNorm2d-11           [-1, 16, 32, 32]              32
       BasicBlock-12           [-1, 16, 32, 32]               0
           Conv2d-13           [-1, 16, 32, 32]           2,304

In [6]:
# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        progress_bar(batch_idx, len(trainloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                     % (train_loss/(batch_idx+1), 100.*correct/total, correct, total))


def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar(batch_idx, len(testloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d))'
                         % (test_loss/(batch_idx+1), 100.*correct/total, correct, total))

    # Save checkpoint.
    acc = 100.*correct/total
    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

In [7]:
for epoch in range(start_epoch, start_epoch+200):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0
  Step: 263ms | Tot: 23s983ms | Loss: 1.758 | Acc: 34.152% (17076/50000) 391/391 
  Step: 12ms | Tot: 1s360ms | Loss: 1.631 | Acc: 41.010% (4101/10000) 100/100 0 
Saving..

Epoch: 1
  Step: 62ms | Tot: 23s731ms | Loss: 1.261 | Acc: 54.358% (27179/50000) 391/391 
  Step: 13ms | Tot: 1s363ms | Loss: 1.209 | Acc: 56.290% (5629/10000) 100/100 
Saving..

Epoch: 2
  Step: 59ms | Tot: 23s459ms | Loss: 1.008 | Acc: 64.112% (32056/50000) 391/391 
  Step: 13ms | Tot: 1s369ms | Loss: 1.184 | Acc: 58.560% (5856/10000) 100/100 
Saving..

Epoch: 3
  Step: 60ms | Tot: 23s485ms | Loss: 0.855 | Acc: 69.884% (34942/50000) 391/391 
  Step: 12ms | Tot: 1s364ms | Loss: 0.920 | Acc: 68.430% (6843/10000) 100/100 
Saving..

Epoch: 4
  Step: 61ms | Tot: 23s586ms | Loss: 0.730 | Acc: 74.532% (37266/50000) 391/391 
  Step: 12ms | Tot: 1s379ms | Loss: 0.825 | Acc: 72.910% (7291/10000) 100/100 
Saving..

Epoch: 5
  Step: 60ms | Tot: 23s547ms | Loss: 0.654 | Acc: 77.234% (38617/50000) 391/391 
  Step: 12m

  Step: 59ms | Tot: 23s452ms | Loss: 0.186 | Acc: 93.614% (46807/50000) 391/391 
  Step: 13ms | Tot: 1s363ms | Loss: 0.442 | Acc: 86.390% (8639/10000) 100/100 

Epoch: 94
  Step: 61ms | Tot: 23s446ms | Loss: 0.189 | Acc: 93.418% (46709/50000) 391/391 
  Step: 12ms | Tot: 1s360ms | Loss: 0.411 | Acc: 87.410% (8741/10000) 100/100 /100 

Epoch: 95
  Step: 59ms | Tot: 23s456ms | Loss: 0.186 | Acc: 93.468% (46734/50000) 391/391 
  Step: 13ms | Tot: 1s370ms | Loss: 0.367 | Acc: 88.390% (8839/10000) 100/100  70/100 

Epoch: 96
  Step: 59ms | Tot: 23s509ms | Loss: 0.188 | Acc: 93.330% (46665/50000) 391/391 
  Step: 13ms | Tot: 1s364ms | Loss: 0.396 | Acc: 88.430% (8843/10000) 100/100  17/10 51/10 72/100 77/100 

Epoch: 97
  Step: 61ms | Tot: 23s566ms | Loss: 0.188 | Acc: 93.400% (46700/50000) 391/391 
  Step: 13ms | Tot: 1s368ms | Loss: 0.423 | Acc: 87.470% (8747/10000) 100/100 

Epoch: 98
  Step: 59ms | Tot: 23s585ms | Loss: 0.190 | Acc: 93.356% (46678/50000) 391/391 
  Step: 12ms | Tot: 1s37

  Step: 13ms | Tot: 1s340ms | Loss: 0.331 | Acc: 92.700% (9270/10000) 100/100 0 

Epoch: 186
  Step: 60ms | Tot: 23s230ms | Loss: 0.007 | Acc: 99.858% (49929/50000) 391/391 
  Step: 13ms | Tot: 1s346ms | Loss: 0.331 | Acc: 92.700% (9270/10000) 100/100  25/100 

Epoch: 187
  Step: 60ms | Tot: 23s567ms | Loss: 0.007 | Acc: 99.850% (49925/50000) 391/391 
  Step: 13ms | Tot: 1s363ms | Loss: 0.331 | Acc: 92.710% (9271/10000) 100/100 

Epoch: 188
  Step: 60ms | Tot: 23s544ms | Loss: 0.008 | Acc: 99.806% (49903/50000) 391/391 
  Step: 12ms | Tot: 1s350ms | Loss: 0.330 | Acc: 92.700% (9270/10000) 100/100 100 95/100 

Epoch: 189
  Step: 60ms | Tot: 23s542ms | Loss: 0.007 | Acc: 99.840% (49920/50000) 391/391 
  Step: 13ms | Tot: 1s353ms | Loss: 0.335 | Acc: 92.540% (9254/10000) 100/100  23/100 

Epoch: 190
  Step: 61ms | Tot: 23s631ms | Loss: 0.007 | Acc: 99.856% (49928/50000) 391/391 
  Step: 13ms | Tot: 1s368ms | Loss: 0.332 | Acc: 92.700% (9270/10000) 100/100 100 

Epoch: 191
  Step: 59ms | T

In [8]:
print('Accuracy:', round(best_acc, 2))
print('Error:', round(100-best_acc, 2))

Accuracy: 92.89
Error: 7.11
